<a href="https://colab.research.google.com/github/DenisBoytsov41/tutors-sentiment-coursework/blob/main/notebooks/03_prepare_labeling_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03_build_target_dataset

Цель ноутбука:
- взять подготовленные артефакты из `02_data_preparation`;
- определить, какие наборы входят в основной целевой контур эксперимента;
- собрать финальный датасет под гипотезу;
- при необходимости подготовить небольшой доменный поднабор на ручную разметку;
- сформировать итоговые `train / validation / test` наборы для следующих этапов.

На этом этапе:
- модель ещё не обучается;
- baseline ещё не запускается;
- финальные метрики ещё не считаются.

Основная логика этапа:
1. загрузить cleaned-артефакты;
2. проверить их состав и структуру;
3. определить основной и вспомогательный контур данных;
4. собрать целевой supervised dataset;
5. подготовить разбиение на `train / validation / test`.

In [31]:
!pip -q install pandas pyarrow matplotlib scikit-learn

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

In [34]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## Пути проекта
На этом шаге подключаем папки, которые уже были сформированы на предыдущих этапах.

In [35]:
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/tutors_sentiment_project")

INTERIM_DIR = DRIVE_PROJECT_ROOT / "02_data_interim"
UNIFIED_CORPUS_DIR = INTERIM_DIR / "unified_corpus"
CLEANED_DIR = INTERIM_DIR / "cleaned"
QC_REPORTS_DIR = INTERIM_DIR / "qc_reports"

DATASETS_READY_DIR = DRIVE_PROJECT_ROOT / "04_datasets_ready"
TRAIN_DIR = DATASETS_READY_DIR / "train"
VALIDATION_DIR = DATASETS_READY_DIR / "validation"
TEST_DIR = DATASETS_READY_DIR / "test"
METADATA_DIR = DATASETS_READY_DIR / "metadata"

for folder in [TRAIN_DIR, VALIDATION_DIR, TEST_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("INTERIM_DIR       :", INTERIM_DIR)
print("UNIFIED_CORPUS_DIR:", UNIFIED_CORPUS_DIR)
print("CLEANED_DIR       :", CLEANED_DIR)
print("DATASETS_READY_DIR:", DATASETS_READY_DIR)

INTERIM_DIR       : /content/drive/MyDrive/tutors_sentiment_project/02_data_interim
UNIFIED_CORPUS_DIR: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/unified_corpus
CLEANED_DIR       : /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned
DATASETS_READY_DIR: /content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready


## Вспомогательные функции
На этом этапе нам важно посмотреть, какие артефакты сохранились после `02_data_preparation`.

In [36]:
def list_files(folder: Path, suffixes=(".csv", ".parquet", ".json")):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in suffixes])

def display_folder_files(folder: Path, title: str):
    files = list_files(folder)
    print("=" * 90)
    print(title)
    print("folder:", folder)
    print("num_files:", len(files))
    for p in files:
        print(" -", p.relative_to(folder))
    return files

def find_first_matching_file(folder: Path, name_candidates, prefer_parquet=False):
    files = list_files(folder)

    if not files:
        return None

    normalized_files = []
    for p in files:
        normalized_name = p.name.lower().replace("-", "_").replace(" ", "_")
        normalized_files.append((p, normalized_name))

    normalized_candidates = [
        c.lower().replace("-", "_").replace(" ", "_")
        for c in name_candidates
    ]

    # идём по кандидатам по порядку приоритета
    for cand in normalized_candidates:
        candidate_matches = [p for p, norm_name in normalized_files if cand in norm_name]

        if candidate_matches:
            candidate_matches = sorted(candidate_matches)

            if prefer_parquet:
                parquet_matches = [p for p in candidate_matches if p.suffix.lower() == ".parquet"]
                if parquet_matches:
                    return parquet_matches[0]

            return candidate_matches[0]

    return None

def read_table(path: Path):
    if path is None:
        return None
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    elif path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Неподдерживаемый формат файла: {path}")

## Инвентаризация артефактов после `02_data_preparation`
Сначала посмотрим, какие файлы лежат в `unified_corpus` и `cleaned`.

In [37]:
unified_files = display_folder_files(UNIFIED_CORPUS_DIR, "Файлы в unified_corpus")
cleaned_files = display_folder_files(CLEANED_DIR, "Файлы в cleaned")

Файлы в unified_corpus
folder: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/unified_corpus
num_files: 2
 - all_sources_standardized.csv
 - all_sources_standardized.parquet
Файлы в cleaned
folder: /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned
num_files: 20
 - domain_aux_pool_prepared.csv
 - domain_aux_pool_prepared.parquet
 - domain_dialogues_aux_prepared.csv
 - domain_dialogues_aux_prepared.parquet
 - domain_dialogues_core_prepared.csv
 - domain_dialogues_core_prepared.parquet
 - domain_dialogues_prepared.csv
 - domain_dialogues_prepared.parquet
 - domain_text_pool_prepared.csv
 - domain_text_pool_prepared.parquet
 - education_feedback_document_level.csv
 - education_feedback_document_level.parquet
 - education_feedback_long.csv
 - education_feedback_long.parquet
 - emotion_aux_prepared.csv
 - emotion_aux_prepared.parquet
 - local_domain_template_prepared.csv
 - local_domain_template_prepared.parquet
 - sentiment_core_prepared.csv
 - sentime

## Поиск ключевых cleaned-артефактов
Нас сейчас интересуют прежде всего:
- общий unified слой;
- sentiment core;
- emotion auxiliary;
- education feedback;
- domain dialogues core;
- domain text pool;
- auxiliary domain pool;
- local domain template.

In [38]:
artifact_candidates = {
    "unified_all_sources": [
        "_all_sources_standardized",
        "all_sources_standardized",
        "all_sources",
        "unified_all_sources",
        "unified_corpus"
    ],
    "sentiment_core_prepared": [
        "sentiment_core_prepared",
        "sentiment_core"
    ],
    "emotion_aux_prepared": [
        "emotion_aux_prepared",
        "emotion_aux"
    ],
    "education_feedback_prepared": [
        "education_feedback_document_level",
        "education_feedback_prepared",
        "education_feedback"
    ],
    "education_feedback_long_prepared": [
        "education_feedback_long_prepared",
        "education_feedback_long"
    ],
    "domain_dialogues_core_prepared": [
        "domain_dialogues_core_prepared",
        "domain_dialogues_core"
    ],
    "domain_dialogues_aux_prepared": [
        "domain_dialogues_aux_prepared",
        "domain_dialogues_aux"
    ],
    "domain_dialogues_prepared": [
        "domain_dialogues_prepared",
        "domain_dialogues"
    ],
    "domain_text_pool": [
        "domain_text_pool_prepared",
        "domain_text_pool"
    ],
    "domain_aux_pool": [
        "domain_aux_pool_prepared",
        "domain_aux_pool"
    ],
    "local_domain_template_prepared": [
        "local_domain_template_prepared",
        "local_domain_template"
    ]
}

artifact_paths = {}

for artifact_name, candidates in artifact_candidates.items():
    if artifact_name == "unified_all_sources":
        artifact_paths[artifact_name] = find_first_matching_file(
            UNIFIED_CORPUS_DIR,
            candidates,
            prefer_parquet=True
        )
    else:
        artifact_paths[artifact_name] = find_first_matching_file(
            CLEANED_DIR,
            candidates,
            prefer_parquet=False
        )

artifact_paths_df = pd.DataFrame([
    {
        "artifact_name": k,
        "path": str(v) if v is not None else None,
        "exists": v is not None
    }
    for k, v in artifact_paths.items()
])

display(artifact_paths_df)

,artifact_name,path,exists
0,unified_all_sources,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/unified_corpus/all_sources_standardized.parquet,True
1,sentiment_core_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/sentiment_core_prepared.csv,True
2,emotion_aux_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/emotion_aux_prepared.csv,True
3,education_feedback_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/education_feedback_document_level.csv,True
4,education_feedback_long_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/education_feedback_long.csv,True
5,domain_dialogues_core_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_dialogues_core_prepared.csv,True
6,domain_dialogues_aux_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_dialogues_aux_prepared.csv,True
7,domain_dialogues_prepared,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_dialogues_prepared.csv,True
8,domain_text_pool,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_text_pool_prepared.csv,True
9,domain_aux_pool,/content/drive/MyDrive/tutors_sentiment_project/02_data_interim/cleaned/domain_aux_pool_prepared.csv,True


## Загрузка ключевых cleaned-артефактов
Если файл найден, загружаем его в DataFrame.

In [39]:
loaded_artifacts = {}

for artifact_name, path in artifact_paths.items():
    if path is not None:
        loaded_artifacts[artifact_name] = read_table(path)
    else:
        loaded_artifacts[artifact_name] = None

for artifact_name, df in loaded_artifacts.items():
    if df is None:
        print(f"{artifact_name}: NOT FOUND")
    else:
        print(f"{artifact_name}: shape={df.shape}")

unified_all_sources: shape=(121708, 14)
sentiment_core_prepared: shape=(34331, 14)
emotion_aux_prepared: shape=(9410, 14)
education_feedback_prepared: shape=(1274, 14)
education_feedback_long_prepared: shape=(24206, 14)
domain_dialogues_core_prepared: shape=(66045, 14)
domain_dialogues_aux_prepared: shape=(400, 14)
domain_dialogues_prepared: shape=(66445, 14)
domain_text_pool: shape=(67319, 14)
domain_aux_pool: shape=(400, 14)
local_domain_template_prepared: shape=(2, 14)


## Базовая проверка структуры загруженных артефактов
Смотрим:
- размеры таблиц;
- названия колонок;
- первые строки.

In [40]:
for artifact_name, df in loaded_artifacts.items():
    print("\n" + "=" * 100)
    print("ARTIFACT:", artifact_name)

    if df is None:
        print("Файл не найден")
        continue

    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.head(3))


ARTIFACT: unified_all_sources
shape: (121708, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,<NA>,<NA>,<NA>,Прорвём информационную блокаду изнутри.,Прорвём информационную блокаду изнутри.,neutral,neutral,sentiment_3class_candidate,{}
1,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,<NA>,<NA>,<NA>,"Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.","Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.",negative,negative,sentiment_3class_candidate,{}
2,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,<NA>,<NA>,<NA>,"Кури-и тебя не укусит злая собака, потому что ты воняешь,кури-и тебя не ограбят,так как ты испугаешь грабителей отвратительным громким кашлем, кури - и ты не умрешь от старости, просто не доживешь...","Кури-и тебя не укусит злая собака, потому что ты воняешь,кури-и тебя не ограбят,так как ты испугаешь грабителей отвратительным громким кашлем, кури - и ты не умрешь от старости, просто не доживешь...",skip,<NA>,sentiment_3class_candidate,{}



ARTIFACT: sentiment_core_prepared
shape: (34331, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,NaN,NaN,NaN,Прорвём информационную блокаду изнутри.,Прорвём информационную блокаду изнутри.,neutral,neutral,sentiment_3class_candidate,{}
1,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,NaN,NaN,NaN,"Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.","Никогда у меня не будет ""одного приложения для всего"". В топку эти ущербные универсальности.",negative,negative,sentiment_3class_candidate,{}
2,rusentiment,sentiment_core,core,rusentiment_preselected_posts.csv,unknown,NaN,NaN,NaN,"Есть 3 типа людей: Умные, которые делают все сразу. Умные, которые откладывают на потом и доделывают. И Я...сначала впадлу потом поздно","Есть 3 типа людей: Умные, которые делают все сразу. Умные, которые откладывают на потом и доделывают. И Я...сначала впадлу потом поздно",neutral,neutral,sentiment_3class_candidate,{}



ARTIFACT: emotion_aux_prepared
shape: (9410, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


...

,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,student_feedback_ru,domain_education,domain,test.csv,test,NaN,NaN,student,"Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ...","Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т..."
1,student_feedback_ru,domain_education,domain,test.csv,test,NaN,NaN,student,"Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем.","Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем.",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 0, ""видео-уроки"": 0, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 0, ""практики__семинары"": 0, ""т..."
2,student_feedback_ru,domain_education,domain,test.csv,test,NaN,NaN,student,"Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в...","Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в...",NaN,NaN,education_aspect_document,"{""aspect_labels"": {""лекции"": 0, ""доклады"": 0, ""проекты"": 0, ""презентации"": 0, ""фильмы"": 1, ""видео-уроки"": 1, ""задания__задачи"": 0, ""онлайн-курс"": 0, ""баллы__оценки"": 1, ""практики__семинары"": 0, ""т..."



ARTIFACT: domain_aux_pool
shape: (400, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,instructional_dialogues_ru,domain_dialogue_aux,aux_domain,instructional_dialogues_ru.csv,unknown,1,0,student,Как мне topic 1?,Как мне topic 1?,NaN,NaN,unlabeled_dialogue,"{""topic"": ""Topic 1"", ""goal"": ""Пользователь узнаёт, как topic 1.""}"
1,instructional_dialogues_ru,domain_dialogue_aux,aux_domain,instructional_dialogues_ru.csv,unknown,1,1,tutor_analog,"Вот шаги, как topic 1.","Вот шаги, как topic 1.",NaN,NaN,unlabeled_dialogue,"{""topic"": ""Topic 1"", ""goal"": ""Пользователь узнаёт, как topic 1.""}"
2,instructional_dialogues_ru,domain_dialogue_aux,aux_domain,instructional_dialogues_ru.csv,unknown,1,2,student,Спасибо!,Спасибо!,NaN,NaN,unlabeled_dialogue,"{""topic"": ""Topic 1"", ""goal"": ""Пользователь узнаёт, как topic 1.""}"



ARTIFACT: local_domain_template_prepared
shape: (2, 14)
columns: ['dataset_name', 'source_group', 'priority', 'source_file', 'split', 'conversation_id', 'turn_id', 'speaker_role', 'text_raw', 'text_clean', 'raw_label', 'target_label', 'label_type', 'meta_json']


,dataset_name,source_group,priority,source_file,split,conversation_id,turn_id,speaker_role,text_raw,text_clean,raw_label,target_label,label_type,meta_json
0,local_domain_ru,local_domain,future_core,local_domain_ru_template.csv,template,conv_0001,0,student,"Мне сложно понять, как выбрать следующую дисциплину.","Мне сложно понять, как выбрать следующую дисциплину.",NaN,NaN,unlabeled_dialogue,"{""meta"": NaN}"
1,local_domain_ru,local_domain,future_core,local_domain_ru_template.csv,template,conv_0001,1,tutor,Давай посмотрим на твою цель и текущую траекторию.,Давай посмотрим на твою цель и текущую траекторию.,NaN,NaN,unlabeled_dialogue,"{""meta"": NaN}"


## Контроль ключевых полей
Проверяем, что в основных таблицах есть хотя бы базовые поля:
- `dataset_name`
- `text_clean`
- `target_label` (где должна быть supervised-разметка)
- `speaker_role`
- `conversation_id`
- `turn_id`

In [41]:
key_columns = ["dataset_name", "text_clean", "target_label", "speaker_role", "conversation_id", "turn_id"]

check_rows = []

for artifact_name, df in loaded_artifacts.items():
    if df is None:
        continue

    row = {"artifact_name": artifact_name, "rows": len(df)}
    for col in key_columns:
        row[col] = col in df.columns
    check_rows.append(row)

structure_check_df = pd.DataFrame(check_rows)
display(structure_check_df)

,artifact_name,rows,dataset_name,text_clean,target_label,speaker_role,conversation_id,turn_id
0,unified_all_sources,121708,True,True,True,True,True,True
1,sentiment_core_prepared,34331,True,True,True,True,True,True
2,emotion_aux_prepared,9410,True,True,True,True,True,True
3,education_feedback_prepared,1274,True,True,True,True,True,True
4,education_feedback_long_prepared,24206,True,True,True,True,True,True
5,domain_dialogues_core_prepared,66045,True,True,True,True,True,True
6,domain_dialogues_aux_prepared,400,True,True,True,True,True,True
7,domain_dialogues_prepared,66445,True,True,True,True,True,True
8,domain_text_pool,67319,True,True,True,True,True,True
9,domain_aux_pool,400,True,True,True,True,True,True


## Первая содержательная сводка
Сейчас нам важно увидеть:
- какие артефакты уже содержат `target_label`;
- какие артефакты являются доменно близкими, но пока неразмеченными;
- сколько в них строк.

In [42]:
summary_rows = []

for artifact_name, df in loaded_artifacts.items():
    if df is None:
        continue

    row = {
        "artifact_name": artifact_name,
        "rows": len(df),
        "has_target_label_column": "target_label" in df.columns,
        "non_null_target_label": int(df["target_label"].notna().sum()) if "target_label" in df.columns else 0,
        "has_speaker_role": "speaker_role" in df.columns,
        "has_conversation_id": "conversation_id" in df.columns,
        "has_turn_id": "turn_id" in df.columns,
        "num_datasets_inside": int(df["dataset_name"].nunique()) if "dataset_name" in df.columns else None
    }
    summary_rows.append(row)

artifact_summary_df = pd.DataFrame(summary_rows).sort_values(by="rows", ascending=False)
display(artifact_summary_df)

,artifact_name,rows,has_target_label_column,non_null_target_label,has_speaker_role,has_conversation_id,has_turn_id,num_datasets_inside
0,unified_all_sources,121708,True,34331,True,True,True,7
8,domain_text_pool,67319,True,0,True,True,True,2
7,domain_dialogues_prepared,66445,True,0,True,True,True,2
5,domain_dialogues_core_prepared,66045,True,0,True,True,True,1
1,sentiment_core_prepared,34331,True,34331,True,True,True,2
4,education_feedback_long_prepared,24206,True,0,True,True,True,1
2,emotion_aux_prepared,9410,True,0,True,True,True,1
3,education_feedback_prepared,1274,True,0,True,True,True,1
6,domain_dialogues_aux_prepared,400,True,0,True,True,True,1
9,domain_aux_pool,400,True,0,True,True,True,1


## Контроль по классам в sentiment-ядре
Эта ячейка нужна, чтобы понять, действительно ли `sentiment_core_prepared` уже готов как основа для 3-классовой задачи.

In [43]:
sentiment_core_df = loaded_artifacts.get("sentiment_core_prepared")

if sentiment_core_df is None:
    print("sentiment_core_prepared не найден")
else:
    print("shape:", sentiment_core_df.shape)
    if "target_label" in sentiment_core_df.columns:
        display(
            sentiment_core_df["target_label"]
            .fillna("<NA>")
            .value_counts(dropna=False)
            .rename_axis("target_label")
            .reset_index(name="count")
        )

    if "dataset_name" in sentiment_core_df.columns:
        display(
            sentiment_core_df.groupby(["dataset_name", "target_label"])
            .size()
            .reset_index(name="count")
            .sort_values(["dataset_name", "count"], ascending=[True, False])
        )

shape: (34331, 14)


,target_label,count
0,neutral,18061
1,positive,9060
2,negative,7210


,dataset_name,target_label,count
1,rusentiment,neutral,12720
2,rusentiment,positive,6646
0,rusentiment,negative,3912
4,rusentitweet,neutral,5341
3,rusentitweet,negative,3298
5,rusentitweet,positive,2414


## Контроль доменного пула
Посмотрим, из чего состоит `domain_text_pool`.

In [44]:
domain_text_pool_df = loaded_artifacts.get("domain_text_pool")

if domain_text_pool_df is None:
    print("domain_text_pool не найден")
else:
    print("shape:", domain_text_pool_df.shape)

    if "dataset_name" in domain_text_pool_df.columns:
        display(
            domain_text_pool_df["dataset_name"]
            .value_counts(dropna=False)
            .rename_axis("dataset_name")
            .reset_index(name="count")
        )

    preview_cols = [c for c in ["dataset_name", "speaker_role", "text_clean", "target_label"] if c in domain_text_pool_df.columns]
    display(domain_text_pool_df[preview_cols].head(10))

shape: (67319, 14)


,dataset_name,count
0,teacher_student_dialogues_ru,66045
1,student_feedback_ru,1274


,dataset_name,speaker_role,text_clean,target_label
0,student_feedback_ru,student,"Электив хороший, преподаватель тоже, но для меня полезного было не так много. Я записалась на него в тот момент, когда на одной из профильных дисциплин мы как раз изучали нейронки, причём в таком ...",NaN
1,student_feedback_ru,student,"Хороший преподаватель, интересные и полезные знания дает в этой области. Электив достаточно легкий, закрыться может каждый без лишних проблем.",NaN
2,student_feedback_ru,student,"Важно обязательно ходить на пары. Чтоб вас, как минимум, запомнила учительница. И не сидеть на них в телефоне, хоть иногда пару слов по теме кидать…. Иначе вы не закроете этот электив. И в конце в...",NaN
3,student_feedback_ru,student,"Отличный электив! Преподаватель - умная, начитанная, любит работать со студентами, любит свою работу в целом. Курс позволяет изучить лексику по теме психология и логопедия, ну и соответственно узн...",NaN
4,student_feedback_ru,student,"преподша- Марандина Елена Леонидовна. Электив проходил в соцгуме. Электив нормальный, смотрите презентации, записываете лекции и иногда делаете мини-проверочные. По ощущениям-как подготовка к ОГЭ ...",NaN
5,student_feedback_ru,student,"Вам это не надо, не берите этот электив",NaN
6,student_feedback_ru,student,"Здесь одни семинары, без высшего участия тут пара не пройдёт, но самое главное - интересно. Педагог максимально пытается заявлечь, и у неё это получается. Она максимально показывает свое мастерств...",NaN
7,student_feedback_ru,student,"Хорошая возможность узнать о странах побольше! Очень доброжелательный преподаватель, критика только для помощи! На элективе есть разные варианты набора баллов, что очень удобно.",NaN
8,student_feedback_ru,student,"Очень рада, что выбрала именно этот электив. Прекраснейшая преподавательница, даёт реально много полезной информации, вдохновляет. На парах было много практики говорения, именно Speaking CAE. Со в...",NaN
9,student_feedback_ru,student,"Многим курс понравился, в восторге от него. Но лично для меня он не совсем оправдал ожидания. Кто любит поговорить на темы о психологии, вам туда и получите максимальные баллы. Кто не особо открыт...",NaN
